# 스트레스 점수 예측 V15

**Repeated-CV Multi-Seed Quantile ExtraTrees**

- Public 최고 모델의 파생변수와 ExtraTrees 구조 유지
- 3개 시드의 예측을 평균 또는 중앙값으로 안정화
- 두 가지 5-Fold 분할에서 분위수와 집계 방식을 비교
- 두 분할 모두에서 기준 모델보다 좋아진 후보만 선택
- 반올림을 적용하지 않고 연속 예측값을 그대로 제출
- Test 데이터는 전처리 학습이나 후보 선택에 사용하지 않음

In [1]:
from pathlib import Path
import os

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

## 1. 설정

In [2]:
DATA_DIR = Path("/Users/bitsaem/Desktop/stress_project/open")
OUTPUT_DIR = Path("/Users/bitsaem/Desktop/stress_project")

TARGET = "stress_score"
ID_COLUMN = "ID"
MODEL_SEEDS = (42, 77, 2026)
CV_SEEDS = (42, 2026)
N_ESTIMATORS = 1200
QUANTILES = (0.500, 0.505, 0.510, 0.515)
AGGREGATIONS = ("mean", "median")

WEIGHTED_FEATURES = [
    "mean_working", "bmi", "cholesterol", "height",
    "glucose", "weight", "cholesterol_glucose_ratio",
    "bone_density",
]

## 2. 데이터와 행 단위 파생변수

In [3]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

if set(train.columns) - {TARGET} != set(test.columns):
    raise ValueError("Train과 Test 입력 컬럼이 다릅니다.")
if list(submission.columns) != [ID_COLUMN, TARGET]:
    raise ValueError("제출 양식 컬럼이 올바르지 않습니다.")
if not submission[ID_COLUMN].equals(test[ID_COLUMN]):
    raise ValueError("제출 양식과 Test의 ID 순서가 다릅니다.")


def add_features(frame):
    result = frame.copy()
    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]
    result["bmi"] = result["weight"] / height_m.pow(2)
    result["pulse_pressure"] = sbp - dbp
    result["mean_arterial_pressure"] = (sbp + 2.0 * dbp) / 3.0
    result["blood_pressure_ratio"] = sbp / (dbp + 1e-6)
    result["cholesterol_glucose_ratio"] = (
        result["cholesterol"] / (result["glucose"] + 1e-6)
    )
    result["missing_count"] = result.isna().sum(axis=1)
    for feature in WEIGHTED_FEATURES:
        result[f"{feature}__copy1"] = result[feature]
        result[f"{feature}__copy2"] = result[feature]
    return result


y = train[TARGET].reset_index(drop=True)
X = add_features(
    train.drop(columns=[TARGET, ID_COLUMN])
).reset_index(drop=True)
X_test = add_features(
    test.drop(columns=[ID_COLUMN])
).reset_index(drop=True)
print("Train:", X.shape, "| Test:", X_test.shape)

Train: (3000, 38) | Test: (3000, 38)


## 3. Train 전용 전처리

In [4]:
def make_preprocessor(frame):
    categorical = frame.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()
    numerical = frame.select_dtypes(include=["number"]).columns.tolist()
    return ColumnTransformer(
        [
            (
                "categorical",
                Pipeline(
                    [
                        (
                            "imputer",
                            SimpleImputer(
                                strategy="constant",
                                fill_value="missing",
                            ),
                        ),
                        (
                            "encoder",
                            OneHotEncoder(
                                handle_unknown="ignore",
                                sparse_output=False,
                            ),
                        ),
                    ]
                ),
                categorical,
            ),
            (
                "numerical",
                SimpleImputer(strategy="median", add_indicator=True),
                numerical,
            ),
        ]
    )


def tree_matrix(model, matrix):
    return np.column_stack(
        [tree.predict(matrix) for tree in model.estimators_]
    )


def aggregate_seed_predictions(seed_predictions, method):
    stacked = np.column_stack(seed_predictions)
    if method == "mean":
        result = stacked.mean(axis=1)
    elif method == "median":
        result = np.median(stacked, axis=1)
    else:
        raise ValueError(method)
    return np.clip(result, 0.0, 1.0)

## 4. 두 가지 5-Fold 분할에서 후보 검증

In [5]:
split_predictions = {}
split_targets = {}

for cv_seed in CV_SEEDS:
    cv = KFold(n_splits=5, shuffle=True, random_state=cv_seed)
    candidate_oof = {
        (quantile, aggregation): np.zeros(len(y))
        for quantile in QUANTILES
        for aggregation in AGGREGATIONS
    }
    base_oof = np.zeros(len(y))

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X), start=1):
        preprocessor = make_preprocessor(X.iloc[train_idx])
        X_train_fold = preprocessor.fit_transform(X.iloc[train_idx])
        X_valid_fold = preprocessor.transform(X.iloc[valid_idx])

        matrices = {}
        for model_seed in MODEL_SEEDS:
            model = ExtraTreesRegressor(
                n_estimators=N_ESTIMATORS,
                min_samples_leaf=1,
                max_features=1,
                random_state=model_seed,
                n_jobs=-1,
            )
            model.fit(X_train_fold, y.iloc[train_idx])
            matrices[model_seed] = tree_matrix(model, X_valid_fold)

        base_oof[valid_idx] = np.clip(
            np.quantile(matrices[42], 0.510, axis=1),
            0.0,
            1.0,
        )
        for quantile in QUANTILES:
            per_seed = [
                np.quantile(matrices[seed], quantile, axis=1)
                for seed in MODEL_SEEDS
            ]
            for aggregation in AGGREGATIONS:
                candidate_oof[(quantile, aggregation)][valid_idx] = (
                    aggregate_seed_predictions(per_seed, aggregation)
                )
        print(f"CV seed {cv_seed} | Fold {fold}: complete")

    split_predictions[cv_seed] = {
        "base": base_oof,
        **candidate_oof,
    }
    split_targets[cv_seed] = y.to_numpy()
    print(
        f"CV seed {cv_seed} base MAE:",
        f"{mean_absolute_error(y, base_oof):.6f}",
    )

CV seed 42 | Fold 1: complete
CV seed 42 | Fold 2: complete
CV seed 42 | Fold 3: complete
CV seed 42 | Fold 4: complete
CV seed 42 | Fold 5: complete
CV seed 42 base MAE: 0.149110
CV seed 2026 | Fold 1: complete
CV seed 2026 | Fold 2: complete
CV seed 2026 | Fold 3: complete
CV seed 2026 | Fold 4: complete
CV seed 2026 | Fold 5: complete
CV seed 2026 base MAE: 0.145614


## 5. 반복 검증에서 안정적인 후보 선택

In [6]:
rows = []
for quantile in QUANTILES:
    for aggregation in AGGREGATIONS:
        maes = []
        improvements = []
        for cv_seed in CV_SEEDS:
            target = split_targets[cv_seed]
            base_prediction = split_predictions[cv_seed]["base"]
            candidate = split_predictions[cv_seed][
                (quantile, aggregation)
            ]
            base_mae = mean_absolute_error(target, base_prediction)
            candidate_mae = mean_absolute_error(target, candidate)
            maes.append(candidate_mae)
            improvements.append(base_mae - candidate_mae)
        rows.append(
            {
                "quantile": quantile,
                "aggregation": aggregation,
                "mean_CV_MAE": np.mean(maes),
                "CV_MAE_std": np.std(maes),
                "mean_improvement": np.mean(improvements),
                "wins": sum(value > 0 for value in improvements),
                "worst_improvement": min(improvements),
            }
        )

results = pd.DataFrame(rows).sort_values(
    ["wins", "worst_improvement", "mean_CV_MAE"],
    ascending=[False, False, True],
).reset_index(drop=True)

# 두 CV 분할에서 모두 기준 모델을 이긴 경우에만 다중 시드 사용
eligible = results[
    (results["wins"] == len(CV_SEEDS))
    & (results["worst_improvement"] > 0)
]
if len(eligible):
    selected = eligible.iloc[0]
    use_multiseed = True
else:
    selected = pd.Series(
        {
            "quantile": 0.510,
            "aggregation": "seed42",
            "mean_CV_MAE": np.mean(
                [
                    mean_absolute_error(
                        split_targets[s],
                        split_predictions[s]["base"],
                    )
                    for s in CV_SEEDS
                ]
            ),
            "wins": 0,
            "worst_improvement": 0.0,
        }
    )
    use_multiseed = False

print("Selected quantile:", selected["quantile"])
print("Selected aggregation:", selected["aggregation"])
print("Repeated-CV mean MAE:", f"{selected['mean_CV_MAE']:.6f}")
print("Wins across CV splits:", int(selected["wins"]))
print(
    "Worst split improvement:",
    f"{selected['worst_improvement']:+.6f}",
)
results

Selected quantile: 0.5
Selected aggregation: median
Repeated-CV mean MAE: 0.147102
Wins across CV splits: 2
Worst split improvement: +0.000250


,quantile,aggregation,mean_CV_MAE,CV_MAE_std,mean_improvement,wins,worst_improvement
0,0.500,median,0.147102,0.001758,0.000261,2,0.000250
1,0.505,mean,0.147108,0.001732,0.000254,2,0.000238
2,0.505,median,0.147102,0.001704,0.000260,2,0.000216
3,0.500,mean,0.147112,0.001785,0.000250,2,0.000213
4,0.515,median,0.147029,0.001591,0.000333,2,0.000176
5,0.510,mean,0.147151,0.001674,0.000211,2,0.000137
6,0.510,median,0.147111,0.001628,0.000251,2,0.000131
7,0.515,mean,0.147152,0.001656,0.000210,2,0.000119


## 6. 전체 Train 학습 및 Test 예측

In [7]:
preprocessor = make_preprocessor(X)
X_train_full = preprocessor.fit_transform(X)
X_test_full = preprocessor.transform(X_test)

final_matrices = {}
required_seeds = MODEL_SEEDS if use_multiseed else (42,)
for model_seed in required_seeds:
    model = ExtraTreesRegressor(
        n_estimators=N_ESTIMATORS,
        min_samples_leaf=1,
        max_features=1,
        random_state=model_seed,
        n_jobs=-1,
    )
    model.fit(X_train_full, y)
    final_matrices[model_seed] = tree_matrix(model, X_test_full)
    print(f"Final seed {model_seed}: complete")

selected_quantile = float(selected["quantile"])
if use_multiseed:
    per_seed_test = [
        np.quantile(
            final_matrices[seed],
            selected_quantile,
            axis=1,
        )
        for seed in MODEL_SEEDS
    ]
    pred_v15 = aggregate_seed_predictions(
        per_seed_test,
        selected["aggregation"],
    )
else:
    pred_v15 = np.clip(
        np.quantile(
            final_matrices[42],
            0.510,
            axis=1,
        ),
        0.0,
        1.0,
    )

print("Prediction range:", pred_v15.min(), pred_v15.max())
print("첫 10개 예측값:", pred_v15[:10])

Final seed 42: complete
Final seed 77: complete
Final seed 2026: complete
Prediction range: 0.0 1.0
첫 10개 예측값: [0.58  0.97  0.19  0.44  0.53  0.455 0.78  0.47  0.51  0.88 ]


## 7. 제출 파일 저장

In [8]:
submission_v15 = submission.copy()
submission_v15[ID_COLUMN] = test[ID_COLUMN]
submission_v15[TARGET] = pred_v15

output_path = OUTPUT_DIR / "submit_v15_repeated_cv_multiseed.csv"
submission_v15.to_csv(output_path, index=False, encoding="utf-8")

print("Saved:", output_path)
print("Rows:", len(submission_v15))
submission_v15.head(10)

Saved: /Users/bitsaem/Desktop/stress_project/submit_v15_repeated_cv_multiseed.csv
Rows: 3000


,ID,stress_score
0,TEST_0000,0.580
1,TEST_0001,0.970
2,TEST_0002,0.190
3,TEST_0003,0.440
4,TEST_0004,0.530
5,TEST_0005,0.455
6,TEST_0006,0.780
7,TEST_0007,0.470
8,TEST_0008,0.510
9,TEST_0009,0.880


## 제출 판단 기준

- `Wins across CV splits: 2`이고 `Worst split improvement`가 양수이면 V15 제출 후보
- 둘 중 하나라도 충족하지 못하면 V15는 자동으로 기존 최고 모델과 동일한 방식으로 복귀
- 기존 Public 최고 제출 파일은 반드시 별도로 보관